In [ ]:
import pandas as pd

# EXTRACCIÓN: Cargamos las tablas relevantes
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

# AUDITORÍA: Volumen y duplicados
tablas = {
    'orders': orders,
    'items': items,
    'customers': customers,
    'products': products,
    'reviews': reviews
}
print("=== RESUMEN DE TABLAS EN BRUTO ===")
for nombre, df in tablas.items():
    print(f"{nombre.upper():<10} -> Filas: {df.shape[0]:<7} | Columnas: {df.shape[1]:<2} | Duplicados exactos: {df.duplicated().sum()}")

# CHEQUEO: Integridad referencial
items_sin_pedido = ~items['order_id'].isin(orders['order_id'])
print("\nÍtems huérfanos (sin registro en orders):", items_sin_pedido.sum())

# CHEQUEO: Estados de los pedidos
print("\n=== ESTADOS DE LOS PEDIDOS ===")
print(orders['order_status'].value_counts())

=== RESUMEN DE TABLAS EN BRUTO ===
ORDERS     -> Filas: 99441   | Columnas: 8  | Duplicados exactos: 0
ITEMS      -> Filas: 112650  | Columnas: 7  | Duplicados exactos: 0
CUSTOMERS  -> Filas: 99441   | Columnas: 5  | Duplicados exactos: 0
PRODUCTS   -> Filas: 32951   | Columnas: 9  | Duplicados exactos: 0
REVIEWS    -> Filas: 99224   | Columnas: 7  | Duplicados exactos: 0

Ítems huérfanos (sin registro en orders): 0

=== ESTADOS DE LOS PEDIDOS ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [2]:
# TRANSFORMACIÓN 1: TABLA ORDERS
# Convertir las columnas de fechas a formato datetime
cols_fechas = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in cols_fechas:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Métricas de logística:
# 1. Días que tarda en llegar el pedido
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
# 2. Diferencia entre fecha estimada y fecha real
orders['delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
# 3. Marca binaria de retraso
orders['is_delayed'] = (orders['delay_days'] > 0).astype(int)

# TRANSFORMACIÓN 2: TABLA PRODUCTS
# Rellenar categorías vacías con 'unknown' para no perder ventas al hacer JOIN
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# TRANSFORMACIÓN 3: TABLA REVIEWS
# Si un pedido tiene más de una reseña, nos quedamos con la más reciente para no duplicar montos al hacer JOIN
reviews_clean = reviews.sort_values('review_creation_date').drop_duplicates(subset=['order_id'], keep='last')

# CARGA: Exportar las tablas limpias
orders.to_csv('../data/processed/orders_clean.csv', index=False)
items.to_csv('../data/processed/items_clean.csv', index=False)
customers.to_csv('../data/processed/customers_clean.csv', index=False)
products.to_csv('../data/processed/products_clean.csv', index=False)
reviews_clean.to_csv('../data/processed/reviews_clean.csv', index=False)

print("Proceso ETL completo. Tablas limpias guardadas en data/processed/.")

Proceso ETL completo. Tablas limpias guardadas en data/processed/.
